# Tattersalls Autumn Horses in Training Sale — Data Preparation

**Notebook 01 of 3** | *TFM — Predictive Modelling of the Tattersalls Horses in Training Market*

This notebook transforms the raw Tattersalls catalogue CSV into a clean, analytically framed
dataset ready for EDA and modelling. Three invariants are enforced throughout:

1. **Reproducibility** — all transformations are deterministic and fully parameterised.
2. **Non-destructive cleaning** — original text columns are preserved; derived fields are added alongside them.
3. **Explicit outcome definitions** — `withdrawn`, `not sold`, `vendor buyback`, and
   `sold to third party` are treated as economically distinct states.

| Step | Section |
|------|---------|
| 0 | Setup & Analytical Constants |
| 1 | Data Ingestion |
| 2 | Column Standardisation |
| 3 | Data Dictionary |
| 4 | Entity Normalisation |
| 5 | Derived Features |
| 6 | Outcome Engineering |
| 7 | Temporal Sanity Check |
| 8 | Export |

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
import plotly.express as px
import onspy

# Global plot settings
sns.set_style("whitegrid")
plt.rcParams['axes.formatter.limits'] = (-20, 20)
plt.rcParams['axes.formatter.useoffset'] = False
#!uv pip install -r requirements.txt --prerelease=allow

> **Design rationale:** This notebook is intentionally strict about three things before interpreting any business pattern: reproducibility, non-destructive cleaning, and explicit target definitions. The goal is not only to describe the Tattersalls market, but to identify which variables are real, which ones are proxies, and which ones would create leakage or misleading conclusions in a later modeling stage.
**Note**: To model the market accurately, we define `df_offered` by explicitly excluding 'withdrawn' lots. Withdrawn lots never faced the market and thus shouldn't be treated as failures to sell. We also strictly separate 'vendor buybacks' from actual 'sales to third parties' to avoid upward price bias.

## 0. Setup & Analytical Constants

Global imports, plot defaults, and domain constants.
The `PREMIUM_CUTOFF` (150 000 gns) defines the elite-tier threshold used throughout the TFM.
CPI values (ONS, October reference month, base 2024) enable real-price comparisons across
the full 2009–2025 window. Utility functions for bootstrapped CIs and permutation tests
are defined here once and reused in the EDA notebook.

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
# Global Analytical Constants
PREMIUM_CUTOFF = 150000  # 150k guineas: market threshold for high-value lots
EARLY_PERIOD = (2009, 2015)
MID_PERIOD = (2016, 2020)
RECENT_PERIOD = (2021, 2025)
cpi_october = {
    2009: 89.2, 2010: 91.9, 2011: 96.5, 2012: 99.1, 2013: 101.3, 2014: 102.6,
    2015: 102.6, 2016: 103.5, 2017: 106.6, 2018: 109.2, 2019: 110.8, 2020: 111.8,
    2021: 116.5, 2022: 129.4, 2023: 135.5, 2024: 138.6, 2025: 141.4  # Estimate for 2025
}
BASE_YEAR = 2024
COUNTRY_SUFFIX_RE = re.compile(r'\s*\(([A-Z]+)\)\s*$')
def extract_country_suffix(series: pd.Series) -> pd.Series:
    """Extract country codes such as (GB) without destroying the original string."""
    return series.astype('string').str.extract(COUNTRY_SUFFIX_RE, expand=False)
def strip_country_suffix(series: pd.Series) -> pd.Series:
    """Remove country suffixes such as (GB) while keeping the full entity name."""
    return series.astype('string').str.replace(COUNTRY_SUFFIX_RE, '', regex=True).str.strip()
def parse_numeric_series(series: pd.Series) -> pd.Series:
    """Parse Tattersalls-style numeric fields like 90.000 or 108.675 into floats."""
    cleaned = (
        series.astype('string')
        .str.strip()
        .replace({'-': pd.NA, 'nan': pd.NA, 'None': pd.NA, '': pd.NA}) # type: ignore
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )
    return pd.to_numeric(cleaned, errors='coerce')
def title_from_canonical(series: pd.Series) -> pd.Series:
    """Convert canonical uppercase labels into a readable title-case display string."""
    return (
        series.astype('string')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.lower()
        .str.title()
    )
def normalize_root_entity(name: str, stopwords=None, aliases=None):
    """Conservative normalization for high-cardinality entity names."""
    if pd.isna(name):
        return None
    stopwords = stopwords or set()
    aliases = aliases or {}
    s = str(name).upper().strip()
    s = aliases.get(s, s)
    s = s.replace('&', ' AND ')
    s = re.sub(r'[^A-Z0-9\s]', ' ', s)
    tokens = [t for t in s.split() if t not in stopwords]
    s = ' '.join(tokens)
    s = re.sub(r'\s+', ' ', s).strip()
    return s if s else None
def bootstrap_ci(values, stat_func=np.median, n_boot=2000, ci=0.95, random_state=42):
    """
    Bootstrap confidence interval for a univariate statistic.
    Vectorized implementation for efficiency with large samples.
    Parameters:
    -----------
    values : array-like
        Data to bootstrap
    stat_func : callable
        Statistic function (default: np.median for robustness with heavy tails)
    n_boot : int
        Number of bootstrap samples (default: 2000)
    ci : float
        Confidence level (default: 0.95)
    random_state : int
        Random seed for reproducibility
    Returns:
    --------
    tuple : (observed_statistic, ci_lower, ci_upper)
    Notes:
    ------
    Uses percentile method. For n > 4000 with median or proportions,
    this converges to BCa method (Efron & Tibshirani, 1993).
    For small samples (n < 500) with asymmetric distributions,
    consider using scipy.stats.bootstrap with method='BCa'.
    """
    clean = pd.Series(values).dropna().astype(float).to_numpy()
    if clean.size == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(random_state)
    # Vectorized sampling: generate all bootstrap samples at once
    # Shape: (n_boot, clean.size)
    boot_samples = rng.choice(clean, size=(n_boot, clean.size), replace=True)
    # Apply statistic to each bootstrap sample
    # Optimization: use axis-specific functions when possible (avoids apply_along_axis overhead)
    if stat_func == np.median:
        boot_stats = np.median(boot_samples, axis=1)
    elif stat_func == np.mean:
        boot_stats = np.mean(boot_samples, axis=1)
    elif stat_func == np.std:
        boot_stats = np.std(boot_samples, axis=1, ddof=1)
    else:
        # Fallback for custom functions (e.g., trimmed mean, quantiles)
        boot_stats = np.apply_along_axis(stat_func, 1, boot_samples)
    observed = stat_func(clean)
    alpha = (1 - ci) / 2
    return observed, np.quantile(boot_stats, alpha), np.quantile(boot_stats, 1 - alpha)
def bootstrap_proportion_ci(boolean_values, n_boot=2000, ci=0.95, random_state=42):
    """
    Bootstrap confidence interval for a binary proportion.
    Vectorized implementation using binomial sampling optimization.
    Parameters:
    -----------
    boolean_values : array-like
        Binary data (0/1 or True/False)
    n_boot : int
        Number of bootstrap samples (default: 2000)
    ci : float
        Confidence level (default: 0.95)
    random_state : int
        Random seed for reproducibility
    Returns:
    --------
    tuple : (observed_proportion, ci_lower, ci_upper)
    Notes:
    ------
    For proportions with n > 5000, the percentile method is 
      approximately equivalent to BCa due to CLT convergence.
     Mathematical note: The bootstrap distribution of a proportion 
      is approximately symmetric when n*p and n*(1-p) > 10.
    """
    clean = pd.Series(boolean_values).dropna().astype(int).to_numpy()
    if clean.size == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(random_state)
    # Vectorized approach: generate all samples at once
    # Shape: (n_boot, clean.size)
    boot_samples = rng.choice(clean, size=(n_boot, clean.size), replace=True)
    # Calculate proportion for each bootstrap sample
    # Sum along axis=1 gives count of 1s, divide by n gives proportion
    boot_props = boot_samples.sum(axis=1) / clean.size
    observed_prop = clean.mean()
    alpha = (1 - ci) / 2
    return observed_prop, np.quantile(boot_props, alpha), np.quantile(boot_props, 1 - alpha)
def permutation_test(values_a, values_b, stat_func=np.median, n_perm=5000, random_state=42):
    """
    Two-sided permutation test for a difference in statistics.
    Tests the null hypothesis that two groups have the same distribution 
     (specifically, that their statistics are equal).
    Parameters:
    ----------
    values_a : array-like
        First group data
    values_b : array-like
        Second group data
    stat_func : callable
        Statistic function to compare (default: np.median for robustness)
    n_perm : int
        Number of permutations (default: 5000)
    random_state : int
        Random seed for reproducibility
    Returns:
    --------
    tuple : (observed_difference, p_value)
    Notes:
    -----
    The permutation test is exact (not approximate) under H₀,
     making it more robust than parametric tests for heavy-tailed data.
    Assumptions:
     - Exchangeability under H₀: if groups come from the same distribution,
        any permutation is equally likely
     - Independence within and between groups
    For heavy-tailed data with different tail behaviors,
     consider using robust statistics like median or trimmed mean.
    """
    a = pd.Series(values_a).dropna().astype(float).to_numpy()
    b = pd.Series(values_b).dropna().astype(float).to_numpy()
    if a.size == 0 or b.size == 0:
        return np.nan, np.nan
    # Observed difference in statistics between groups
    observed_diff = stat_func(a) - stat_func(b)
    # Pool both groups for permutation under H₀ (null hypothesis: no difference)
    pooled = np.concatenate([a, b])
    rng = np.random.default_rng(random_state)
    # Vectorized permutation approach would require storing all permuted arrays,
    # which is memory-intensive. Loop is acceptable here.
    perm_stats = np.empty(n_perm)
    for i in range(n_perm):
        shuffled_indices = rng.permutation(len(pooled))
        perm_a = pooled[shuffled_indices[:len(a)]]
        perm_b = pooled[shuffled_indices[len(a):]]
        perm_stats[i] = stat_func(perm_a) - stat_func(perm_b)
    # Two-sided p-value: proportion of permutations with absolute difference >= observed
    p_value = np.mean(np.abs(perm_stats) >= abs(observed_diff))
    return observed_diff, p_value
def mean_annual_share_table(df, entity_col, label_col=None):
    """
    Normalize entity prominence by annual market share instead of raw counts.
    This approach corrects for year-to-year variation in total sales volume,
    giving a more accurate picture of entity prominence over time.
    Parameters:
    ----------
    df : DataFrame
        Input data with 'sale_year' and entity_col columns
    entity_col : str
        Column name for entity identifier (e.g., 'buyer_normalized')
    label_col : str, optional
        Column name for display label (e.g., 'buyer_title')
    Returns:
    -------
    DataFrame with columns:
        - label_col (if provided): most common display name for entity
        - total_sales: sum of sales across all years
        - active_years: number of years with at least one sale
        - mean_annual_share: average share within each year
        - peak_annual_share: maximum share in any single year
    Notes:
    -----
    Mean annual share is more robust than total share when comparing entities 
      across periods with different total volumes.
    Example interpretation:
        If mean_annual_share = 0.15, the entity averaged 15% of market 
         activity in years they were active.
    """
    per_year = (
        df.groupby(['sale_year', entity_col])
        .size()
        .rename('sales')
        .reset_index()
    )
    # Calculate share within each year (corrects for volume variation)
    per_year['share_within_year'] = per_year['sales'] / per_year.groupby('sale_year')['sales'].transform('sum')
    summary = (
        per_year.groupby(entity_col)
        .agg(
            total_sales=('sales', 'sum'),
            active_years=('sale_year', 'nunique'),
            mean_annual_share=('share_within_year', 'mean'),
            peak_annual_share=('share_within_year', 'max')
        )
        .sort_values(['mean_annual_share', 'total_sales'], ascending=[False, False])
    )
    if label_col is not None:
        # Get most common label for each entity (handles minor variations)
        label_map = (
            df[[entity_col, label_col]]
            .dropna()
            .groupby(entity_col)[label_col]
            .agg(lambda x: x.value_counts().index[0])
        )
        summary.insert(0, label_col, summary.index.to_series().map(label_map))
    return summary


## 1. Data Ingestion

Load the raw Tattersalls CSV (26 076 lots, 2009–2025).
The file uses European numeric formatting (`90.000` for ninety thousand) and
mixed-language column names — both are resolved in subsequent steps.

In [3]:
autumn_horses_df = pd.read_csv('../data/Autumn Horses In Training Sale 2009-2024.csv')
autumn_horses_df.info()
autumn_horses_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 26076 entries, 0 to 26075
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Day             26076 non-null  int64  
 1   Lot             26076 non-null  str    
 2   Name            26076 non-null  str    
 3   Sex             26054 non-null  str    
 4   Colour          26053 non-null  str    
 5   Sire            26049 non-null  str    
 6   Dam             26049 non-null  str    
 7   Year Foaled     26054 non-null  float64
 8   Date Foaled     26054 non-null  str    
 9   Grandsire       26024 non-null  str    
 10  Damsire         26024 non-null  str    
 11  Covered by      6 non-null      str    
 12  Consignor       26049 non-null  str    
 13  Purchaser       26076 non-null  str    
 14  Price (gns)     18695 non-null  str    
 15  Stabling        454 non-null    str    
 16  Año Subasta     26076 non-null  int64  
 17  Nombre Subasta  26076 non-null  str    
 1

,Day,Lot,Name,Sex,Colour,Sire,Dam,Year Foaled,Date Foaled,Grandsire,Damsire,Covered by,Consignor,Purchaser,Price (gns),Stabling,Año Subasta,Nombre Subasta,Price (€),ORIG,SIRE_N,DAM_N,SIREDAM_N,BREEDER_N
0,3,1067,Qirat (GB),G,Ch,Showcasing (GB),Emulous (GB),"2,021.00",22/2/2021,Oasis Dream (GB),Dansili (GB),NaN,Juddmonte,Lot Withdrawn,NaN,NaN,2024,Autumn Horses In Training Sale 2024,-,2021SHOWCASINGEMULOUS,SHOWCASING,EMULOUS,DANSILI,JUDDMONTE
1,2,639,Gifted Master (IRE),G,B,Kodiac (GB),Shobobb (GB),"2,013.00",3/4/2013,Danehill (USA),Shamardal (USA),NaN,The Castlebridge Consignment,Lot Withdrawn,NaN,NaN,2019,Autumn Horses in Training Sale 2019,-,2013KODIACSHOBOBB,KODIAC,SHOBOBB,SHAMARDAL,THE CASTLEBRIDGE CONSIGNMENT
2,2,568,Commanche Falls (GB),G,Br,Lethal Force (IRE),Joyeaux (GB),"2,017.00",28/4/2017,Dark Angel (IRE),Mark of Esteem (IRE),NaN,Denton Hall Stables (M. Dods),Lot Withdrawn,NaN,NaN,2020,Autumn Horses In Training Sale 2020,-,2017LETHAL FORCEJOYEAUX,LETHAL FORCE,JOYEAUX,MARK OF ESTEEM,DENTON HALL STABLES
3,2,764,Summerghand (IRE),G,B,Lope de Vega (IRE),Kate The Great (GB),"2,014.00",6/3/2014,Shamardal (USA),Xaar (GB),NaN,David O'Meara Racing Ltd.,Vendor,90.000,NaN,2020,Autumn Horses In Training Sale 2020,108.675,2014LOPE DE VEGAKATE THE GREAT,LOPE DE VEGA,KATE THE GREAT,XAAR,DAVID O'MEARA RACING LTD.
4,2,691,Regal Reality (GB),G,B,Intello (GER),Regal Realm (GB),"2,015.00",20/2/2015,Galileo (IRE),Medicean (GB),NaN,Freemason Lodge Stables (Sir M. Stoute),Lot Withdrawn,NaN,NaN,2020,Autumn Horses In Training Sale 2020,-,2015INTELLOREGAL REALM,INTELLO,REGAL REALM,MEDICEAN,FREEMASON LODGE STABLES


## 2. Column Standardisation

Three sequential passes:
1. Strip leading/trailing whitespace from column names.
2. Lowercase all names for uniform access.
3. Rename domain-specific labels to unambiguous `snake_case` identifiers
   (e.g. `Año Subasta` → `sale_year`, `Price (gns)` → `price_gns`).

In [4]:
# Normalise column names
print("Before normalising names:",autumn_horses_df.columns)
col_names_to_normalise = [col for col in autumn_horses_df.columns if col.strip() != col]
for col_name in col_names_to_normalise:
    autumn_horses_df[col_name.strip()] = autumn_horses_df[col_name]
    autumn_horses_df.drop(columns=[col_name],inplace=True)
print("After normalising names:",autumn_horses_df.columns)

Before normalising names: Index(['Day', 'Lot', 'Name', 'Sex', 'Colour', 'Sire', 'Dam', 'Year Foaled',
       'Date Foaled', 'Grandsire', 'Damsire', 'Covered by', 'Consignor',
       'Purchaser', 'Price (gns)', 'Stabling', 'Año Subasta', 'Nombre Subasta',
       ' Price (€)', ' ORIG', ' SIRE_N', ' DAM_N', ' SIREDAM_N', ' BREEDER_N'],
      dtype='str')
After normalising names: Index(['Day', 'Lot', 'Name', 'Sex', 'Colour', 'Sire', 'Dam', 'Year Foaled',
       'Date Foaled', 'Grandsire', 'Damsire', 'Covered by', 'Consignor',
       'Purchaser', 'Price (gns)', 'Stabling', 'Año Subasta', 'Nombre Subasta',
       'Price (€)', 'ORIG', 'SIRE_N', 'DAM_N', 'SIREDAM_N', 'BREEDER_N'],
      dtype='str')


## 3. Data Dictionary

The raw file mixes descriptive catalog metadata, commercial outcomes, and already-normalized entity fields. The key point is that some columns are useful for interpretation but dangerous for modeling if they are not framed correctly.
### Core catalog fields
- `Day`: sale day from 1 to 5. It behaves more like catalogue positioning than pure calendar time.
- `Lot`: lot number inside each sale year.
- `Name`: horse name, often with country suffix.
- `Sex`: `C`, `F`, `G`, `H`, `M`, `R`. These labels encode both biological sex and, indirectly, market segment.
- `Colour`: horse colour.
- `Sire`, `Dam`, `Grandsire`, `Damsire`: pedigree fields. They arrive both as raw labels and, for some of them, as normalized canonical versions.
- `Consignor`: seller or consignor label as written in the catalogue.
- `Covered by`: almost empty in this dataset, so analytically weak unless a very specific niche use case is targeted.
- `Stabling`: also extremely sparse, so it should be treated as auxiliary metadata, not as a central predictor.
### Time and price fields
- `Year Foaled`: foaling year.
- `Date Foaled`: full foaling date. It is **not** redundant here; the field is almost complete and contains real month/day variation that can proxy relative maturity.
- `Año Subasta` (original CSV name): sale year. Renamed to `sale_year`.
- `Nombre Subasta` (original CSV name): sale name. Renamed to `sale_name`.
- `Price (gns)`: price in guineas, the most reliable monetary field for analysis.
- `Price (€)`: euro amount embedded in the source file. It should be treated as descriptive only, not as a historically comparable macro-financial series.
### Outcome and normalization fields
- `Purchaser`: buyer label or administrative outcome (`Lot Withdrawn`, `Lot Not Sold`, `Vendor`). These states are economically different and must not be collapsed blindly. 
- `ORIG`: normalized combination of foaling year, sire, and dam. It is useful for auditing but not safe as a unique key.
- `SIRE_N`, `DAM_N`, `SIREDAM_N`, `BREEDER_N`: canonicalized entity labels already supplied by the source. These are preferable to manual row realignment.

In [5]:
col_names_to_lowercase = autumn_horses_df.columns.str.lower()
autumn_horses_df.columns = col_names_to_lowercase
autumn_horses_df.columns

Index(['day', 'lot', 'name', 'sex', 'colour', 'sire', 'dam', 'year foaled',
       'date foaled', 'grandsire', 'damsire', 'covered by', 'consignor',
       'purchaser', 'price (gns)', 'stabling', 'año subasta', 'nombre subasta',
       'price (€)', 'orig', 'sire_n', 'dam_n', 'siredam_n', 'breeder_n'],
      dtype='str')

In [6]:
columns_to_rename = {
    'name': 'horse_name',
    'year foaled': 'birth_year',
    'date foaled': 'date_foaled',
    'price (gns)': 'price_gns',
    'price (€)': 'price_euros_raw',
    'año subasta': 'sale_year',
    'nombre subasta': 'sale_name',
    'covered by': 'stallion',
    'breeder_n': 'consignor_n',
    'siredam_n': 'damsire_n',
}
autumn_horses_df.rename(columns=columns_to_rename, inplace=True)
# Convert price_gns from European string format to numeric (90.000 → 90000.0)
autumn_horses_df['price_gns'] = parse_numeric_series(autumn_horses_df['price_gns'])

## 4. Entity Normalisation & Cleaning

Pedigree and commercial entity fields (sire, dam, grandsire, damsire, consignor) arrive with
country suffixes `(GB)`, typographic noise, and abbreviation variants.
This section:
- Extracts and stores country codes separately (e.g. `sire_country`).
- Applies conservative root-entity normalisation for consignors
  (removes legal suffixes, resolves known aliases such as `JUDDMONTE FARMS → JUDDMONTE`).
- Normalises the catalogue day number to a 0–1 ratio to handle 4- vs 5-day sale editions.

In [7]:
# clean entities without overwriting the original text columns.
# Clean entities without overwriting original columns.
string_columns = [
    'horse_name', 'sex', 'colour', 'sire', 'dam', 'grandsire', 'damsire',
    'stallion', 'consignor', 'purchaser', 'sale_name', 'orig',
    'sire_n', 'dam_n', 'damsire_n', 'consignor_n', 'price_euros_raw'
]
for col in string_columns:
    if col in autumn_horses_df.columns:
        autumn_horses_df[col] = autumn_horses_df[col].astype('string').str.strip()
entity_columns = ['horse_name', 'sire', 'dam', 'grandsire', 'damsire', 'stallion']
for col in entity_columns:
    autumn_horses_df[f'{col}_country'] = extract_country_suffix(autumn_horses_df[col])
    autumn_horses_df[f'{col}_clean'] = strip_country_suffix(autumn_horses_df[col])
autumn_horses_df['horse_name_clean'] = autumn_horses_df['horse_name_clean'].fillna(autumn_horses_df['horse_name'])
autumn_horses_df['consignor_display'] = autumn_horses_df['consignor'].str.replace(r'\s+', ' ', regex=True).str.strip()
autumn_horses_df['sire_entity'] = autumn_horses_df['sire_n']
autumn_horses_df['dam_entity'] = autumn_horses_df['dam_n']
autumn_horses_df['damsire_entity'] = autumn_horses_df['damsire_n']
autumn_horses_df['consignor_entity_exact'] = autumn_horses_df['consignor_n'].fillna(autumn_horses_df['consignor'].str.upper())
autumn_horses_df['consignor_entity'] = autumn_horses_df['consignor_entity_exact']
consignor_stopwords = {'LTD', 'LIMITED', 'LLP', 'INC', 'COMPANY', 'CO', 'AGENT', 'IRELAND', 'STABLES', 'STABLE', 'RACING'}
consignor_aliases = {'JUDDMONTE FARMS': 'JUDDMONTE'}
autumn_horses_df['sire_clean'] = autumn_horses_df['sire_clean'].fillna(title_from_canonical(autumn_horses_df['sire_entity']))
autumn_horses_df['dam_clean'] = autumn_horses_df['dam_clean'].fillna(title_from_canonical(autumn_horses_df['dam_entity']))
autumn_horses_df['damsire_clean'] = autumn_horses_df['damsire_clean'].fillna(title_from_canonical(autumn_horses_df['damsire_entity']))
autumn_horses_df['consignor_label'] = title_from_canonical(autumn_horses_df['consignor_entity_exact']).fillna(autumn_horses_df['consignor_display'])
# Day Normalization (Equivalence 4 vs 5 days)
# Logic: Map days to a 0.0 - 1.0 scale within each year to ensure comparability.
def get_day_norm_ratio(day_series):
    max_day = day_series.max()
    if max_day <= 1: 
        return 0.0
    return (day_series - 1) / (max_day - 1)
autumn_horses_df['day_norm_ratio'] = autumn_horses_df.groupby('sale_year')['day'].transform(get_day_norm_ratio)
# To keep it categorical and readable: "Day 1", "Day 2", "Day 3", "Day 4"
# We'll use a 4-bin approach or just handle the 5th day as "Late Sale"
autumn_horses_df['day_normalized'] = np.round(autumn_horses_df['day_norm_ratio'] * 3 + 1).astype(int)
autumn_horses_df['consignor_family'] = autumn_horses_df['consignor_entity_exact'].map(
    lambda x: normalize_root_entity(x, stopwords=consignor_stopwords, aliases=consignor_aliases)
).fillna(autumn_horses_df['consignor_label'])
autumn_horses_df['consignor_clean'] = autumn_horses_df['consignor_label']
autumn_horses_df['consignor_model'] = autumn_horses_df['consignor_entity_exact']
autumn_horses_df['birth_year'] = autumn_horses_df['birth_year'].astype('Int64')
autumn_horses_df['sale_year'] = autumn_horses_df['sale_year'].astype('Int64')
autumn_horses_df['date_foaled'] = pd.to_datetime(autumn_horses_df['date_foaled'], dayfirst=True, errors='coerce')
autumn_horses_df[[
    'horse_name', 'horse_name_clean', 'horse_name_country',
    'grandsire', 'grandsire_clean', 'grandsire_country',
    'consignor', 'consignor_label', 'consignor_family'
]].head()

,horse_name,horse_name_clean,horse_name_country,grandsire,grandsire_clean,grandsire_country,consignor,consignor_label,consignor_family
0,Qirat (GB),Qirat,GB,Oasis Dream (GB),Oasis Dream,GB,Juddmonte,Juddmonte,JUDDMONTE
1,Gifted Master (IRE),Gifted Master,IRE,Danehill (USA),Danehill,USA,The Castlebridge Consignment,The Castlebridge Consignment,THE CASTLEBRIDGE CONSIGNMENT
2,Commanche Falls (GB),Commanche Falls,GB,Dark Angel (IRE),Dark Angel,IRE,Denton Hall Stables (M. Dods),Denton Hall Stables,DENTON HALL
3,Summerghand (IRE),Summerghand,IRE,Shamardal (USA),Shamardal,USA,David O'Meara Racing Ltd.,David O'Meara Racing Ltd.,DAVID O MEARA
4,Regal Reality (GB),Regal Reality,GB,Galileo (IRE),Galileo,IRE,Freemason Lodge Stables (Sir M. Stoute),Freemason Lodge Stables,FREEMASON LODGE


## 5. Derived Features

All derived columns are additive — no source column is deleted. Key derivations:
- `age_at_sale`: sale year minus birth year.
- `foaled_month` / `foaled_quarter`: seasonality proxy for relative maturity within a cohort.
- `is_late_catalogue_day`: boolean flag for catalogue positions on days 4–5.
- `sire_dam_combo`: cross-reference key for pedigree novelty scoring in Feature Engineering.

In [8]:
# derive features that are analytically useful and reproducible.
autumn_horses_df['age_at_sale'] = autumn_horses_df['sale_year'] - autumn_horses_df['birth_year']
autumn_horses_df['foaled_month'] = autumn_horses_df['date_foaled'].dt.month.astype('Int64')
autumn_horses_df['foaled_quarter'] = autumn_horses_df['date_foaled'].dt.quarter.astype('Int64')
autumn_horses_df['is_late_catalogue_day'] = autumn_horses_df['day'].isin([4, 5])
autumn_horses_df['sire_dam_combo'] = autumn_horses_df['sire_entity'].fillna('UNKNOWN') + '_x_' + autumn_horses_df['dam_entity'].fillna('UNKNOWN')
missing_summary = (
    autumn_horses_df.isna()
    .mean()
    .sort_values(ascending=False)
    .rename('missing_share')
    .to_frame()
)
missing_summary.head(12)

,missing_share
stallion_clean,1.00
stallion,1.00
stallion_country,1.00
stabling,0.98
price_gns,0.28
damsire_country,0.08
grandsire_country,0.05
sire_country,0.00
dam_country,0.00
damsire,0.00


## 6. Outcome Engineering

The `sale_outcome` variable encodes **five mutually exclusive states**:

| Outcome | Description |
|---------|-------------|
| `sold_to_third_party` | Genuine market transaction — the target for price modelling |
| `vendor_buyback` | Reserve met by the vendor; price recorded but no real transfer |
| `not_sold` | Failed to meet any bid at or above reserve |
| `withdrawn` | Never entered the ring; excluded from sell-through denominators |
| `other_or_inconsistent` | Data anomaly flag (should be 0 after cleaning) |

The distinction between `vendor_buyback` and `not_sold` is critical: collapsing them
would inflate apparent clearance rates and introduce upward price bias.

In [9]:
# define outcomes explicitly instead of collapsing all non-sales together.
# Define outcomes explicitly instead of collapsing all non-sales.
withdrawn_mask = autumn_horses_df['purchaser'].eq('Lot Withdrawn')
not_sold_mask = autumn_horses_df['purchaser'].eq('Lot Not Sold')
vendor_mask = autumn_horses_df['purchaser'].eq('Vendor')
sold_to_third_party_mask = (~withdrawn_mask & ~not_sold_mask & ~vendor_mask) & autumn_horses_df['price_gns'].notna()
autumn_horses_df['sale_outcome'] = np.select(
    [withdrawn_mask, not_sold_mask, vendor_mask, sold_to_third_party_mask],
    ['withdrawn', 'not_sold', 'vendor_buyback', 'sold_to_third_party'],
    default='other_or_inconsistent'
)
autumn_horses_df['sale_outcome'] = pd.Categorical(
    autumn_horses_df['sale_outcome'],
    categories=['sold_to_third_party', 'vendor_buyback', 'not_sold', 'withdrawn', 'other_or_inconsistent'],
    ordered=False,
)
autumn_horses_df['has_price_quote'] = autumn_horses_df['price_gns'].notna()
autumn_horses_df['is_offered_for_sale'] = ~withdrawn_mask
autumn_horses_df['sold_to_third_party'] = sold_to_third_party_mask
autumn_horses_df['vendor_buyback'] = vendor_mask
autumn_horses_df['lot_not_sold'] = not_sold_mask
autumn_horses_df['lot_withdrawn'] = withdrawn_mask
autumn_horses_df['sold'] = autumn_horses_df['sold_to_third_party']
autumn_horses_df['log_price_gns'] = np.nan
positive_log_mask = autumn_horses_df['sold_to_third_party'] & autumn_horses_df['price_gns'].gt(0)
autumn_horses_df.loc[positive_log_mask, 'log_price_gns'] = np.log(
    autumn_horses_df.loc[positive_log_mask, 'price_gns']
)
autumn_horses_df['sale_outcome'].value_counts(dropna=False)

sale_outcome
sold_to_third_party      16531
withdrawn                 7081
vendor_buyback            1383
not_sold                  1081
other_or_inconsistent        0
Name: count, dtype: int64

## 7. Temporal Sanity Check

Annual aggregate statistics serve two purposes:
1. A final consistency check — confirming that derived flags sum correctly across years.
2. A summary table that feeds directly into Section 2 of the EDA notebook.

`sale_rate_on_offered` (sold to third party ÷ lots offered) is the primary clearance KPI
used throughout the TFM. `withdrawal_rate` tracks the pre-ring withdrawal trend.

In [10]:
# 2. Temporal structure
by_year = (
    autumn_horses_df.groupby('sale_year')
    .agg(
        total_catalogued=('lot', 'size'),
        offered=('is_offered_for_sale', 'sum'),
        sold_to_third_party=('sold_to_third_party', 'sum'),
        vendor_buyback=('vendor_buyback', 'sum'),
        lot_not_sold=('lot_not_sold', 'sum'),
        withdrawn=('lot_withdrawn', 'sum'),
        median_price_sold=('price_gns', lambda s: s[autumn_horses_df.loc[s.index, 'sold_to_third_party']].median())
    )
)
by_year['sale_rate_on_catalogue'] = 100 * by_year['sold_to_third_party'] / by_year['total_catalogued']
by_year['sale_rate_on_offered'] = 100 * by_year['sold_to_third_party'] / by_year['offered']
by_year['withdrawal_rate'] = 100 * by_year['withdrawn'] / by_year['total_catalogued']
# Merge median_price_real if price_real_gns already exists (defined in cell 26 onward)
if 'price_real_gns' in autumn_horses_df.columns:
    _real_median = (
        autumn_horses_df[autumn_horses_df['sold_to_third_party']]
        .groupby('sale_year')['price_real_gns']
        .median()
        .rename('median_price_real')
    )
    by_year = by_year.join(_real_median)
by_year.round(2)

,total_catalogued,offered,sold_to_third_party,vendor_buyback,lot_not_sold,withdrawn,median_price_sold,sale_rate_on_catalogue,sale_rate_on_offered,withdrawal_rate
sale_year,,,,,,,,,,
2009,1533,1063,903,69,91,470,"9,000.00",58.90,84.95,30.66
2010,1583,1087,865,88,134,496,"9,000.00",54.64,79.58,31.33
2011,1478,1020,848,80,92,458,"9,000.00",57.37,83.14,30.99
2012,1429,1015,909,69,37,414,"11,000.00",63.61,89.56,28.97
2013,1518,1088,894,119,75,430,"10,000.00",58.89,82.17,28.33
2014,1539,1047,922,78,47,492,"13,000.00",59.91,88.06,31.97
2015,1679,1220,1038,92,90,459,"10,000.00",61.82,85.08,27.34
2016,1516,1032,949,52,31,484,"13,500.00",62.60,91.96,31.93
2017,1650,1255,1065,111,79,395,"11,000.00",64.55,84.86,23.94


In [11]:
df_sold = autumn_horses_df.loc[autumn_horses_df['sold_to_third_party']].copy()


## 8. Export

A single Parquet file — `data/processed/clean_data.parquet` — is the source of truth
for all downstream notebooks. Parquet preserves dtypes (including `Int64` nullable integers
and `Categorical`) without round-trip loss.

In [12]:
os.makedirs('../data/processed', exist_ok=True)
autumn_horses_df.to_parquet('../data/processed/clean_data.parquet', index=False)
print('Data saved to data/processed/clean_data.parquet')

Data saved to data/processed/clean_data.parquet
